# Ollama + Cloudflare Tunnel: Serve Fine-Tuned Qwen Medical Validator

Runs the fine-tuned **Qwen 2.5 3B** medical-validator model (GGUF Q4_K_M, ~1.8GB) on a Colab GPU via **Ollama**, then exposes it to your local dev environment through a **Cloudflare Tunnel**.

**Before running:**
1. Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU
2. Have the `medical-validator-q4_k_m.gguf` file on Google Drive or ready to upload

**Every cell is idempotent** — safe to re-run without restarting the runtime.

In [ ]:
# ============================================================
# Configuration — edit these before running
# ============================================================

# Set True to mount Google Drive and copy the GGUF from there.
# Set False to upload the GGUF manually via the Colab file browser.
USE_GOOGLE_DRIVE = True

# Path to the GGUF on your Google Drive (only used if USE_GOOGLE_DRIVE=True)
GDRIVE_GGUF_PATH = "/content/drive/MyDrive/models/medical-validator-q4_k_m.gguf"

# Local working path — the GGUF will be copied/uploaded here
LOCAL_GGUF_PATH = "/content/medical-validator-q4_k_m.gguf"

# Ollama model name (must match what the worker expects)
MODEL_NAME = "medical-validator"

In [ ]:
# ============================================================
# Cell 2: GPU Check + Install Ollama
# ============================================================
import subprocess, shutil

# Verify GPU is available
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("WARNING: No GPU detected!")
    print("Go to Runtime > Change runtime type > T4 GPU, then restart.")
    print("Without a GPU, inference will be extremely slow (~2 min/request).")
    print()
else:
    # Extract GPU name from nvidia-smi output
    for line in gpu_check.stdout.split("\n"):
        if "Tesla" in line or "A100" in line or "V100" in line or "T4" in line or "L4" in line:
            print(f"GPU detected: {line.strip()}")
            break
    else:
        print("GPU detected (check nvidia-smi for details)")
    print()

# Install Ollama (idempotent — reinstalls are fast)
if not shutil.which("ollama"):
    print("Installing Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("Ollama already installed.")

!ollama --version

In [ ]:
# ============================================================
# Cell 3: Load GGUF (Google Drive or manual upload)
# ============================================================
import os, shutil, time

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")

    if not os.path.exists(GDRIVE_GGUF_PATH):
        raise FileNotFoundError(
            f"GGUF not found at {GDRIVE_GGUF_PATH}\n"
            "Upload it to Google Drive first, or set USE_GOOGLE_DRIVE = False."
        )

    # Skip copy if already present and same size
    src_size = os.path.getsize(GDRIVE_GGUF_PATH)
    if os.path.exists(LOCAL_GGUF_PATH) and os.path.getsize(LOCAL_GGUF_PATH) == src_size:
        print(f"GGUF already at {LOCAL_GGUF_PATH} ({src_size / 1024**2:.0f} MB) — skipping copy.")
    else:
        print(f"Copying {src_size / 1024**2:.0f} MB from Drive (this takes ~30-60s)...")
        start = time.time()
        shutil.copy2(GDRIVE_GGUF_PATH, LOCAL_GGUF_PATH)
        elapsed = time.time() - start
        print(f"Done in {elapsed:.0f}s.")
else:
    print("Google Drive mount skipped.")
    if not os.path.exists(LOCAL_GGUF_PATH):
        print()
        print("Upload your GGUF manually:")
        print("  1. Click the folder icon in the left sidebar")
        print("  2. Click the upload button")
        print(f"  3. Upload 'medical-validator-q4_k_m.gguf'")
        print()
        print("Then re-run this cell.")

# Final check
if os.path.exists(LOCAL_GGUF_PATH):
    size_mb = os.path.getsize(LOCAL_GGUF_PATH) / 1024**2
    print(f"\nGGUF ready: {LOCAL_GGUF_PATH} ({size_mb:.0f} MB)")
else:
    raise FileNotFoundError(f"GGUF not found at {LOCAL_GGUF_PATH}")

In [ ]:
# ============================================================
# Cell 4: Start Ollama server + create model
# ============================================================
import subprocess, time, os, signal, urllib.request, urllib.error

# --- Write the Modelfile (not an f-string — avoids template escaping issues) ---
modelfile_lines = [
    f"FROM {LOCAL_GGUF_PATH}",
    "",
    "PARAMETER temperature 0",
    "PARAMETER num_ctx 2048",
    "PARAMETER stop <|im_end|>",
    "",
    'TEMPLATE """{{ if .System }}<|im_start|>system',
    '{{ .System }}<|im_end|>',
    '{{ end }}<|im_start|>user',
    '{{ .Prompt }}<|im_end|>',
    '<|im_start|>assistant',
    '"""',
]
with open("/content/Modelfile", "w") as f:
    f.write("\n".join(modelfile_lines) + "\n")
print("Modelfile written.")

# --- Start ollama serve (kill any existing instance first) ---
def is_ollama_running():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if is_ollama_running():
    print("Ollama server already running.")
else:
    # Kill any zombie process holding the port
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(1)

    print("Starting ollama serve...")
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/content/ollama_serve.log", "w"),
        stderr=subprocess.STDOUT,
        preexec_fn=os.setsid,  # detach so cell interrupts don't kill it
    )

    # Wait for readiness
    for i in range(30):
        if is_ollama_running():
            print(f"Ollama server ready ({i+1}s).")
            break
        time.sleep(1)
    else:
        print("Server log:")
        !cat /content/ollama_serve.log
        raise RuntimeError("Ollama server failed to start within 30s.")

# --- Create model (skip if already registered) ---
check = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL_NAME in check.stdout:
    print(f"Model '{MODEL_NAME}' already registered.")
else:
    print(f"Creating model '{MODEL_NAME}'...")
    result = subprocess.run(
        ["ollama", "create", MODEL_NAME, "-f", "/content/Modelfile"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"STDOUT: {result.stdout}")
        print(f"STDERR: {result.stderr}")
        raise RuntimeError("ollama create failed")
    print("Model created.")

print()
!ollama list

In [ ]:
# ============================================================
# Cell 5: Test the model
# ============================================================
import json, urllib.request, time

payload = json.dumps({
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "system",
            "content": "You are a healthcare entity verification specialist. MUST respond with valid JSON only."
        },
        {
            "role": "user",
            "content": json.dumps({
                "company_name": "Mayo Clinic",
                "jurisdiction_code": "us_mn",
                "company_number": "1A-26",
                "current_status": "Active",
                "incorporation_date": "1919-08-19",
                "company_type": "Non-Profit Corporation"
            })
        }
    ],
    "temperature": 0,
    "stream": False
}).encode("utf-8")

req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"},
)

print("Sending test request...")
start = time.time()
try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode())
        elapsed = time.time() - start
        content = result["choices"][0]["message"]["content"]
        tokens = result.get("usage", {}).get("total_tokens", "?")
        print(f"OK — {elapsed:.1f}s, {tokens} tokens")
        print()
        # Try to pretty-print if valid JSON, otherwise raw
        try:
            print(json.dumps(json.loads(content), indent=2))
        except json.JSONDecodeError:
            print(content)
except Exception as e:
    print(f"FAILED: {e}")
    print("Check that ollama serve is running (re-run Cell 4).")

In [ ]:
# ============================================================
# Cell 6: Expose via Cloudflare Tunnel
# ============================================================
import subprocess, time, re, os, signal

CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"
TUNNEL_LOG = "/content/cloudflared.log"

# --- Install cloudflared if needed ---
if not os.path.exists(CLOUDFLARED_PATH):
    print("Downloading cloudflared...")
    dl = subprocess.run(
        ["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", CLOUDFLARED_PATH],
        capture_output=True, text=True,
    )
    if dl.returncode != 0:
        raise RuntimeError(f"cloudflared download failed: {dl.stderr}")
    os.chmod(CLOUDFLARED_PATH, 0o755)
    print("Installed.")
else:
    print("cloudflared already installed.")
!cloudflared --version

# --- Kill any existing tunnel ---
subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)
time.sleep(1)

# --- Start tunnel ---
print("\nStarting Cloudflare tunnel...")
with open(TUNNEL_LOG, "w") as f:
    pass  # truncate log

tunnel_proc = subprocess.Popen(
    [CLOUDFLARED_PATH, "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=open(TUNNEL_LOG, "a"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

# --- Wait for tunnel URL (up to 60s for slow Colab instances) ---
tunnel_url = None
for i in range(60):
    time.sleep(1)
    try:
        with open(TUNNEL_LOG, "r") as f:
            log = f.read()
        match = re.search(r"(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)", log)
        if match:
            tunnel_url = match.group(1)
            break
    except FileNotFoundError:
        pass

if tunnel_url:
    print()
    print("=" * 64)
    print(f"  Ollama is live!")
    print(f"  QWEN_OLLAMA_URL={tunnel_url}")
    print("=" * 64)
    print()
    print("Paste into your .env:")
    print(f"  QWEN_OLLAMA_URL={tunnel_url}")
    print("  QWEN_CONCURRENCY=5")
    print("  QWEN_TIMEOUT_MS=15000")
    print()
    print(f"Test from your machine:")
    print(f"  curl {tunnel_url}/v1/models")
else:
    print("\nTunnel URL not detected within 60s. Log contents:")
    !cat {TUNNEL_LOG}
    print("\nTry re-running this cell.")

## Keep Alive

Colab disconnects after ~30 min idle (90 min for Pro). Run this cell and **leave the tab open**. Stop it manually when done.

In [ ]:
# ============================================================
# Cell 7: Keep Alive — pings Ollama every 60s
# ============================================================
import time, urllib.request, urllib.error
from datetime import datetime

print("Keepalive started. Press the stop button to end.")
print()

n = 0
while True:
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5)
        status = "OK"
    except Exception as e:
        status = f"FAIL ({e})"

    n += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] #{n}: {status}")
    time.sleep(60)